# LSTM T2D -- SHAP Interpretability Analysis

Corre esto **después** de `LSTM_T2D_Training.ipynb` -- usa el modelo ya entrenado y guardado (`models/lstm_model_regular_1year_best.pth`), no vuelve a entrenar nada.

**Qué hace:**
- Carga el mejor modelo de las 10 iteraciones.
- Explica una muestra balanceada de pacientes (positivos/negativos) con SHAP.
- Cada una de las ~519 features (500 DX + labs/vitals + edad) se trata como un único "jugador" en el juego de Shapley: su contribución es la **suma** de su efecto a lo largo de **todas** las visitas del paciente (no la media, no solo la última visita) -- la Opción C que acordamos, y matemáticamente es exactamente lo que hace un SHAP KernelExplainer cuando "apagar" una feature significa tratarla como no observada (valor=0, máscara=0) en todo el historial a la vez.
- Guarda: gráfica de barras (importancia media), gráfica beeswarm, y los valores SHAP en bruto (CSV) para que puedas rehacer cualquier gráfica sin volver a calcular nada.

**Coste computacional**: SHAP es caro -- por cada paciente que expliques, el modelo se evalúa varios miles de veces (variantes con distintas combinaciones de features "apagadas"). Con el tamaño por defecto (`N_PATIENTS_TO_EXPLAIN = 60`), debería tardar unos minutos en GPU. Si quieres explicar más pacientes, súbelo, pero el tiempo crece linealmente.


In [ ]:
# %% Mount Google Drive and set the working folder
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_FOLDER = "/content/drive/MyDrive/Colab Notebooks/LSTM"
os.chdir(DRIVE_FOLDER)
print("Working directory:", os.getcwd())


In [ ]:
# %% Install packages
!pip install -q --upgrade pandas
!pip install -q shap imbalanced-learn

import pandas as pd
print("pandas version:", pd.__version__)
print("If this is the FIRST time running this cell in this session: Runtime > Restart session, then Runtime > Run all.")


In [ ]:
# %% Imports
import os
import pickle
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence
import shap
import matplotlib.pyplot as plt
import tqdm

def load_streamed_patient_dict(path, desc="Loading"):
    with open(path, "rb") as f:
        first_obj = pickle.load(f)
        if isinstance(first_obj, dict):
            return first_obj
        n_patients = first_obj
        patient_data = {}
        for _ in tqdm.tqdm(range(n_patients), desc=desc, colour='blue'):
            pid, df = pickle.load(f)
            patient_data[pid] = df
        return patient_data

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Using device: {DEVICE}")


In [ ]:
# %% Config -- must match the training notebook's file locations
DATA_PATH = "regular_patients_1_year.pkl"
EARLIEST_DX_PATH = "EARLIEST_DX_deid.csv"
MODEL_PATH = "models/lstm_model_regular_1year_best.pth"
OUTPUT_DIR = "models"  # SHAP outputs saved alongside the training outputs

N_PATIENTS_TO_EXPLAIN = 60   # balanced sample (30 positive + 30 negative, roughly)
KERNEL_SHAP_NSAMPLES = "auto"  # SHAP picks ~2*M+2048 by default; lower this (e.g. 1000) to go faster at the cost of noisier estimates
TOP_N_FEATURES_TO_PLOT = 25

os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
# %% Load the trained model + its architecture config (saved inside the checkpoint)
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
saved_config = checkpoint['config']
print(f"Loaded checkpoint: iteration {checkpoint['iteration']}, test AUC {checkpoint['test_auc']:.4f}")
print(f"Architecture: SEQ_INPUT_SIZE={saved_config['SEQ_INPUT_SIZE']}, "
      f"ORIGINAL_SEQ_SIZE={saved_config['ORIGINAL_SEQ_SIZE']}, DEMO_INPUT_SIZE={saved_config['DEMO_INPUT_SIZE']}")

class ConfigNS:
    pass
config = ConfigNS()
for k, v in saved_config.items():
    setattr(config, k, v)


class PatientRiskModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=config.SEQ_INPUT_SIZE, hidden_size=config.LSTM_HIDDEN_SIZE,
            num_layers=config.NUM_LAYERS, batch_first=True,
            dropout=config.DROPOUT if config.NUM_LAYERS > 1 else 0.0
        )
        self.demo_encoder = nn.Sequential(
            nn.Linear(config.DEMO_INPUT_SIZE, config.DEMO_HIDDEN_SIZE), nn.ReLU(), nn.Dropout(config.DROPOUT)
        )
        self.classifier = nn.Sequential(
            nn.Linear(config.LSTM_HIDDEN_SIZE + config.DEMO_HIDDEN_SIZE, config.COMBINED_HIDDEN_SIZE),
            nn.ReLU(), nn.Dropout(config.DROPOUT), nn.Linear(config.COMBINED_HIDDEN_SIZE, 1)
        )

    def forward(self, x_seq, x_demo, lengths):
        packed_x = pack_padded_sequence(x_seq, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed_x)
        seq_embedding = h_n[-1]
        demo_embedding = self.demo_encoder(x_demo)
        combined = torch.cat((seq_embedding, demo_embedding), dim=1)
        return self.classifier(combined).squeeze(-1)


model = PatientRiskModel(config).to(DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(" Model loaded and ready.")


In [ ]:
# %% Load patient data and rebuild masked feature sequences + labels
# (Same feature-engineering + labeling logic as the training notebook, incl.
# the mislabeling fix -- needed here only to pick a balanced sample of real
# positive/negative patients to explain, not to retrain anything.)
imputed_patients_matrices_all = load_streamed_patient_dict(DATA_PATH, desc="Loading patient data")
df2 = pd.read_csv(EARLIEST_DX_PATH)

df2 = df2.drop(columns=['DX'], errors='ignore')
df2['EARLIEST_DX'] = pd.to_datetime(df2['EARLIEST_DX'], errors='coerce')
df2_earliest = df2.sort_values('EARLIEST_DX').drop_duplicates(subset='PATIENT_ID', keep='first')
assert df2_earliest['PATIENT_ID'].is_unique, "Duplicate PATIENT_IDs after dedup -- DO NOT PROCEED."

imputed_patients = set(imputed_patients_matrices_all.keys())
df2_filtered = df2_earliest[df2_earliest['PATIENT_ID'].isin(imputed_patients)]
dx_date_dict = df2_filtered.set_index('PATIENT_ID')['EARLIEST_DX'].dt.date.to_dict()

first_pid = next(iter(imputed_patients_matrices_all))
all_indices = imputed_patients_matrices_all[first_pid].index
seq_indices = [idx for idx in all_indices if idx not in config.DEMO_COLUMNS]
assert len(seq_indices) == config.ORIGINAL_SEQ_SIZE, \
    f"Feature count mismatch: data has {len(seq_indices)}, model expects {config.ORIGINAL_SEQ_SIZE}. Wrong data file for this model?"

patient_records = {}  # pid -> (masked_features array, label)
for pid, df_patient in tqdm.tqdm(imputed_patients_matrices_all.items(), desc="Building sequences", colour='cyan'):
    demo_features = df_patient.loc[config.DEMO_COLUMNS].iloc[:, 0].values
    seq_raw = df_patient.loc[seq_indices].T.values
    mask = (~np.isnan(seq_raw)).astype(np.float32)
    seq_vals = np.nan_to_num(seq_raw, nan=0.0)
    masked_features = np.concatenate([seq_vals, mask], axis=1)

    visit_dates = [pd.to_datetime(col).date() for col in df_patient.columns]
    diagnosis_date = dx_date_dict.get(pid)
    first_dx_idx = -1
    if diagnosis_date:
        for i, v_date in enumerate(visit_dates):
            if v_date >= diagnosis_date:
                first_dx_idx = i
                break
        else:
            first_dx_idx = len(visit_dates)

    if first_dx_idx != -1:
        if first_dx_idx >= 2:
            patient_records[pid] = (masked_features[:first_dx_idx, :], demo_features, 1)
    else:
        if df_patient.shape[1] >= 2:
            patient_records[pid] = (masked_features, demo_features, 0)

print(f" Built {len(patient_records)} patient records "
      f"({sum(1 for _,_,y in patient_records.values() if y==1)} positive, "
      f"{sum(1 for _,_,y in patient_records.values() if y==0)} negative).")


In [ ]:
# %% Sample a balanced set of patients to explain
rng = np.random.RandomState(42)
pos_ids = [pid for pid, (_, _, y) in patient_records.items() if y == 1]
neg_ids = [pid for pid, (_, _, y) in patient_records.items() if y == 0]
n_each = N_PATIENTS_TO_EXPLAIN // 2
sample_pos = rng.choice(pos_ids, size=min(n_each, len(pos_ids)), replace=False)
sample_neg = rng.choice(neg_ids, size=min(n_each, len(neg_ids)), replace=False)
sample_pids = list(sample_pos) + list(sample_neg)
print(f"Sampled {len(sample_pids)} patients to explain ({len(sample_pos)} positive, {len(sample_neg)} negative).")

feature_names = [str(idx) for idx in seq_indices] + list(config.DEMO_COLUMNS)
N_SEQ = config.ORIGINAL_SEQ_SIZE
M = N_SEQ + config.DEMO_INPUT_SIZE  # total "meta-features" (one per raw feature, whole-history)


In [ ]:
# %% SHAP wrapper: each meta-feature = one raw feature, "on/off" for its WHOLE
# visit history at once. This gives the Option-C semantics (summed
# contribution across all visits) directly and correctly, because turning a
# feature "off" here means the model sees it as fully missing (value=0,
# mask=0) across every timestep in one Shapley coalition -- not something
# summed after the fact.
def make_patient_predict_fn(masked_features, demo_features):
    n_timesteps = masked_features.shape[0]
    lengths = torch.tensor([n_timesteps])

    def f(Z):
        n = Z.shape[0]
        batch_seq = np.zeros((n, n_timesteps, N_SEQ * 2), dtype=np.float32)
        batch_demo = np.zeros((n, config.DEMO_INPUT_SIZE), dtype=np.float32)
        for i in range(n):
            z = Z[i]
            m = masked_features.copy()
            off_features = np.where(z[:N_SEQ] == 0)[0]
            if len(off_features) > 0:
                m[:, off_features] = 0.0            # value columns
                m[:, N_SEQ + off_features] = 0.0    # mask columns -> looks "missing"
            batch_seq[i] = m
            batch_demo[i] = demo_features if z[N_SEQ] == 1 else 0.0

        preds = []
        BATCH = 256
        with torch.no_grad():
            for start in range(0, n, BATCH):
                end = min(start + BATCH, n)
                bs = torch.tensor(batch_seq[start:end]).to(DEVICE)
                bd = torch.tensor(batch_demo[start:end]).to(DEVICE)
                bl = lengths.repeat(end - start)
                out = model(bs, bd, bl)
                preds.append(torch.sigmoid(out).cpu().numpy())
        return np.concatenate(preds)

    return f


In [ ]:
# %% Run SHAP for each sampled patient
background = np.zeros((1, M))
all_shap_values = np.zeros((len(sample_pids), M))
all_feature_display_values = np.zeros((len(sample_pids), M))  # for the beeswarm color axis

for row_i, pid in enumerate(tqdm.tqdm(sample_pids, desc="Computing SHAP per patient", colour='green')):
    masked_features, demo_features, label = patient_records[pid]
    f_patient = make_patient_predict_fn(masked_features, demo_features)
    explainer = shap.KernelExplainer(f_patient, background)
    X_explain = np.ones((1, M))
    sv = explainer.shap_values(X_explain, nsamples=KERNEL_SHAP_NSAMPLES, silent=True)
    all_shap_values[row_i] = np.array(sv).flatten()

    # Display value per feature for the beeswarm plot's color axis: last
    # observed (non-missing) value for sequential features (or 0 if the
    # patient never had it recorded), and the demographic value as-is.
    for j in range(N_SEQ):
        col_vals = masked_features[:, j]
        col_mask = masked_features[:, N_SEQ + j]
        observed = col_vals[col_mask == 1]
        all_feature_display_values[row_i, j] = observed[-1] if len(observed) > 0 else 0.0
    all_feature_display_values[row_i, N_SEQ:] = demo_features

print(" SHAP computation complete.")


In [ ]:
# %% Save raw SHAP values + summary plots
# Output format aligned with the BERT SHAP script (evaluate_and_shap.py) for
# an equitable comparison: same columns (feature/code, mean_shap,
# mean_abs_shap, n_occurrences), same red/blue signed bar plot style.
# One real structural difference remains, unavoidably: BERT aggregates
# across every OCCURRENCE of a code (multiple positions x multiple
# patients), since a code can appear many times in one patient's token
# sequence. The LSTM already collapses each feature to ONE whole-history
# value per patient (that's the Option-C design), so here aggregation is
# across PATIENTS only, and "n_occurrences" means "patients where this
# feature was ever observed" rather than "raw token occurrences".

shap_df = pd.DataFrame(all_shap_values, columns=feature_names)
shap_df.insert(0, 'PATIENT_ID', sample_pids)
shap_path = os.path.join(OUTPUT_DIR, "shap_values_per_patient.csv")
shap_df.to_csv(shap_path, index=False)
print(f" Raw per-patient SHAP values saved to: {shap_path}")

# "n_occurrences" analog: for DX flags, whether ever observed (mask==1 at
# any visit); for continuous labs/vitals, same -- whether ever recorded.
n_patients_present = np.zeros(M, dtype=int)
for row_i, pid in enumerate(sample_pids):
    masked_features, demo_features, label = patient_records[pid]
    for j in range(N_SEQ):
        if masked_features[:, N_SEQ + j].max() > 0:  # mask channel ever 1
            n_patients_present[j] += 1
n_patients_present[N_SEQ:] = len(sample_pids)  # demographics are always "present"

importance_df = pd.DataFrame({
    'feature': feature_names,
    'mean_shap': all_shap_values.mean(axis=0),
    'mean_abs_shap': np.abs(all_shap_values).mean(axis=0),
    'n_occurrences': n_patients_present,
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
importance_path = os.path.join(OUTPUT_DIR, "shap_top_features_regular_1year.csv")
importance_df.to_csv(importance_path, index=False)
print(f" Feature importance ranking saved to: {importance_path}")
print(importance_df.head(10))

# --- Signed bar plot (red = pushes toward T2D, blue = pushes away), same
# style/convention as the BERT script's shap_top_codes_<horizon>.png ---
top = importance_df.head(TOP_N_FEATURES_TO_PLOT).iloc[::-1]
colors = ["#d62728" if v > 0 else "#1f77b4" for v in top['mean_shap']]
fig, ax = plt.subplots(figsize=(8, max(6, TOP_N_FEATURES_TO_PLOT * 0.3)))
ax.barh(top['feature'], top['mean_shap'], color=colors)
ax.set_xlabel("Mean SHAP value (-> T2D risk)")
ax.set_title(f"Top {TOP_N_FEATURES_TO_PLOT} features by |SHAP| -- regular_1year (LSTM)")
ax.axvline(0, color='black', linewidth=0.8)
fig.tight_layout()
bar_path = os.path.join(OUTPUT_DIR, "shap_top_features_regular_1year.png")
fig.savefig(bar_path, dpi=200)
plt.show()
print(f" Signed bar plot saved to: {bar_path}")

# --- Beeswarm plot (extra, not in the BERT script, but useful for the LSTM
# since it also shows how the feature's actual value relates to its impact) ---
top_unsigned = importance_df.head(TOP_N_FEATURES_TO_PLOT)
top_idx = [feature_names.index(f) for f in top_unsigned['feature']]
fig2 = plt.figure(figsize=(8, max(6, TOP_N_FEATURES_TO_PLOT * 0.3)))
shap.summary_plot(
    all_shap_values[:, top_idx], all_feature_display_values[:, top_idx],
    feature_names=[feature_names[i] for i in top_idx], show=False
)
beeswarm_path = os.path.join(OUTPUT_DIR, "shap_beeswarm_regular_1year.png")
plt.tight_layout()
plt.savefig(beeswarm_path, dpi=200, bbox_inches='tight')
plt.show()
print(f" Beeswarm plot saved to: {beeswarm_path}")
